In [12]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import requests
import os

options = Options()
# options.add_argument("--headless")

driver = webdriver.Chrome(options=options)

try:
    wait = WebDriverWait(driver, 20)

    # --------------------------------------------------
    # MAIN PAGE
    # --------------------------------------------------
    driver.get("https://ifsca.gov.in/InformalGuidance/Index")

    wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "tbody tr")
        )
    )

    first_title = wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, "a.points")
        )
    )

    detail_url = first_title.get_attribute("href")

    print("DETAIL URL:")
    print(detail_url)

    # --------------------------------------------------
    # DETAIL PAGE
    # --------------------------------------------------
    driver.get(detail_url)

    pdf_elements = wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "tbody tr td:nth-child(3) a")
        )
    )

    pdf_data = []

    for pdf in pdf_elements:
        pdf_data.append({
            "title": pdf.text.strip(),
            "getfile_url": pdf.get_attribute("href")
        })

    print(f"\nPDFS FOUND: {len(pdf_data)}")

    os.makedirs("downloads", exist_ok=True)

    # --------------------------------------------------
    # SESSION WITH SELENIUM COOKIES
    # --------------------------------------------------
    session = requests.Session()

    for cookie in driver.get_cookies():
        session.cookies.set(
            cookie["name"],
            cookie["value"]
        )

    # --------------------------------------------------
    # PROCESS EACH PDF
    # --------------------------------------------------
    for idx, pdf in enumerate(pdf_data, start=1):

        print("\n" + "=" * 80)
        print("TITLE:", pdf["title"])
        print("GETFILE URL:", pdf["getfile_url"])

        # Open GetFileView page
        driver.get(pdf["getfile_url"])

        # Find iframe containing actual PDF
        iframe = wait.until(
            EC.presence_of_element_located(
                (By.TAG_NAME, "iframe")
            )
        )

        real_pdf_url = iframe.get_attribute("src")

        print("REAL PDF URL:")
        print(real_pdf_url)

        # Download using Selenium session
        response = session.get(
            real_pdf_url,
            headers={
                "Referer": pdf["getfile_url"],
                "User-Agent": "Mozilla/5.0"
            },
            timeout=60
        )

        print("STATUS:", response.status_code)
        print("CONTENT TYPE:", response.headers.get("Content-Type"))

        filename = real_pdf_url.split("fileName=")[-1]

        filepath = os.path.join(
            "downloads",
            filename
        )

        with open(filepath, "wb") as f:
            f.write(response.content)

        print("SAVED:", filepath)

finally:
    driver.quit()

DETAIL URL:
https://ifsca.gov.in/InformalGuidance/Index?MId=zcGvy-Iqfcg=

PDFS FOUND: 2

TITLE: Redacted interpretative letter (response) issued by IFSCA
GETFILE URL: https://ifsca.gov.in/CommonDirect/GetFileView?id=91427247c5628a865846d173bcad7da2&fileName=Interpretive_Letter_20260608_0408.pdf
REAL PDF URL:
https://ifsca.gov.in/CommonDirect/ViewFile?id=91427247c5628a865846d173bcad7da2&fileName=Interpretive_Letter_20260608_0408.pdf
STATUS: 200
CONTENT TYPE: application/pdf
SAVED: downloads/Interpretive_Letter_20260608_0408.pdf

TITLE: Redacted request letter
GETFILE URL: https://ifsca.gov.in/CommonDirect/GetFileView?id=91427247c5628a865846d173bcad88ca&fileName=Request_Letter_20260608_0409.pdf
REAL PDF URL:
https://ifsca.gov.in/CommonDirect/ViewFile?id=91427247c5628a865846d173bcad88ca&fileName=Request_Letter_20260608_0409.pdf
STATUS: 200
CONTENT TYPE: application/pdf
SAVED: downloads/Request_Letter_20260608_0409.pdf


In [15]:
import json

# from langchain_ollama import OllamaLLM
from langchain_community.llms import Ollama 


def ask_mistral_for_response_title(titles):

    llm = Ollama(
        model="mistral:latest",
        temperature=0
    )

    prompt = f"""
You are an automated file filtering system for IFSCA Informal Guidance documents.

The titles below belong to the same Informal Guidance case.

One document is usually:
- applicant request letter
- applicant query
- request letter

Another document is usually:
- IFSCA response
- interpretative letter
- informal guidance letter
- clarification issued by IFSCA

Identify the title corresponding to the IFSCA response document.

Titles:
{json.dumps(titles, indent=2)}

Return ONLY valid JSON.

Example:
{{
    "response_title": "Redacted interpretative letter (response) issued by IFSCA"
}}
"""

    response = llm.invoke(prompt).strip()

    if "```json" in response:
        response = response.split("```json")[1].split("```")[0].strip()

    return json.loads(response)["response_title"]


# --------------------------------------------------------
# Example after scraping
# --------------------------------------------------------

pdf_data = [
    {
        "title": "Redacted interpretative letter (response) issued by IFSCA",
        "url": "pdf_url_1"
    },
    {
        "title": "Redacted request letter",
        "url": "pdf_url_2"
    }
]

titles = [pdf["title"] for pdf in pdf_data]

response_title = ask_mistral_for_response_title(titles)

filtered_pdfs = [
    pdf
    for pdf in pdf_data
    if pdf["title"] == response_title
]

print("\nSelected PDF:")
print(filtered_pdfs[0]["title"])
print(filtered_pdfs[0]["url"])


Selected PDF:
Redacted interpretative letter (response) issued by IFSCA
pdf_url_1


In [17]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from langchain_community.llms import Ollama

import requests
import json
import os


def select_response_pdf(pdfs):

    if len(pdfs) == 1:
        return pdfs[0]

    llm = Ollama(
        model="mistral:latest",
        temperature=0.0
    )

    titles_text = "\n".join(
        [f"{i+1}. {p['title']}" for i, p in enumerate(pdfs)]
    )

    prompt = f"""
You are filtering IFSCA Informal Guidance documents.

The titles below belong to the same Informal Guidance case.

One document is usually:
- applicant request letter
- request letter
- query letter

Another document is usually:
- IFSCA response
- interpretative letter
- informal guidance
- clarification issued by IFSCA

Select ONLY the IFSCA response document.

Titles:
{titles_text}

Return ONLY JSON.

Example:
{{
    "selected_title":"Redacted interpretative letter (response) issued by IFSCA"
}}
"""

    response = llm.invoke(prompt).strip()

    print("\nLLM RESPONSE:")
    print(response)

    if "```json" in response:
        response = response.split("```json")[1].split("```")[0].strip()

    data = json.loads(response)

    selected_title = data["selected_title"]

    for pdf in pdfs:
        if pdf["title"] == selected_title:
            return pdf

    return pdfs[0]


# --------------------------------------------------
# SELENIUM
# --------------------------------------------------

options = Options()
# options.add_argument("--headless")

driver = webdriver.Chrome(options=options)

try:

    wait = WebDriverWait(driver, 20)

    # --------------------------------------------------
    # MAIN PAGE
    # --------------------------------------------------

    driver.get("https://ifsca.gov.in/InformalGuidance/Index")

    wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "tbody tr")
        )
    )

    first_title = wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, "a.points")
        )
    )

    main_title = first_title.text.strip()

    detail_url = first_title.get_attribute("href")

    print("\nMAIN TITLE:")
    print(main_title)

    print("\nDETAIL URL:")
    print(detail_url)

    # --------------------------------------------------
    # DETAIL PAGE
    # --------------------------------------------------

    driver.get(detail_url)

    pdf_elements = wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "tbody tr td:nth-child(3) a")
        )
    )

    pdf_data = []

    for pdf in pdf_elements:

        pdf_data.append({
            "title": pdf.text.strip(),
            "getfile_url": pdf.get_attribute("href")
        })

    print(f"\nPDFS FOUND: {len(pdf_data)}")

    for p in pdf_data:
        print(p["title"])

    # --------------------------------------------------
    # LLM FILTER
    # --------------------------------------------------

    selected_pdf = select_response_pdf(pdf_data)

    print("\nSELECTED PDF:")
    print(selected_pdf["title"])

    pdf_data = [selected_pdf]

    # --------------------------------------------------
    # DOWNLOAD SESSION
    # --------------------------------------------------

    session = requests.Session()

    for cookie in driver.get_cookies():
        session.cookies.set(
            cookie["name"],
            cookie["value"]
        )

    os.makedirs("downloads", exist_ok=True)

    # --------------------------------------------------
    # DOWNLOAD ONLY CHOSEN PDF
    # --------------------------------------------------

    for pdf in pdf_data:

        print("\n" + "=" * 80)
        print("DOWNLOADING:")
        print(pdf["title"])

        driver.get(pdf["getfile_url"])

        iframe = wait.until(
            EC.presence_of_element_located(
                (By.TAG_NAME, "iframe")
            )
        )

        real_pdf_url = iframe.get_attribute("src")

        print("\nREAL PDF URL:")
        print(real_pdf_url)

        response = session.get(
            real_pdf_url,
            headers={
                "Referer": pdf["getfile_url"],
                "User-Agent": "Mozilla/5.0"
            },
            timeout=60
        )

        print("STATUS:", response.status_code)
        print("CONTENT TYPE:", response.headers.get("Content-Type"))

        filename = real_pdf_url.split("fileName=")[-1]

        filepath = os.path.join(
            "downloads",
            filename
        )

        with open(filepath, "wb") as f:
            f.write(response.content)

        print("\nSAVED:")
        print(filepath)

finally:
    driver.quit()


MAIN TITLE:
Informal Guidance request received from Applicant [X] seeking clarifications under Framework of Ship Leasing with respect to the Asset Management Support Service and definition of the Group Entity

DETAIL URL:
https://ifsca.gov.in/InformalGuidance/Index?MId=zcGvy-Iqfcg=

PDFS FOUND: 2
Redacted interpretative letter (response) issued by IFSCA
Redacted request letter

LLM RESPONSE:
{
    "selected_title": "Redacted interpretative letter (response) issued by IFSCA"
}

SELECTED PDF:
Redacted interpretative letter (response) issued by IFSCA

DOWNLOADING:
Redacted interpretative letter (response) issued by IFSCA

REAL PDF URL:
https://ifsca.gov.in/CommonDirect/ViewFile?id=91427247c5628a865846d173bcad7da2&fileName=Interpretive_Letter_20260608_0408.pdf
STATUS: 200
CONTENT TYPE: application/pdf

SAVED:
downloads/Interpretive_Letter_20260608_0408.pdf
